Importação Bibliotecas

In [1]:
from youtube_transcript_api import YouTubeTranscriptApi
import csv
import string

Funções Base

In [16]:
def pega_transcricao(video_id:str , idioma:str='pt'):
    transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
    for item in transcript_list:
        transcript = item.translate(idioma).fetch()
    return transcript


In [17]:
def trata_transcricao(transcript, pause_threshold: int=3):
    # Split into sections and include start and end timestamps
    sections = []
    current_section = ""
    start_time = None
    previous_end = 0
    pause_threshold = pause_threshold

    for line in transcript:
        if current_section and (line["start"] - previous_end > pause_threshold):
            # If there is a pause greater than 5s, we deem the end of section
            end_time = line["start"]
            sections.append(
                {
                    "text": current_section.strip(),
                    "start_time": start_time,
                    "end_time": end_time,
                }
            )
            current_section = ""
            start_time = None
        else:
            # If this is the start of a new section, record the start time
            if not start_time:
                start_time = line["start"]

            # Add the line to the current paragraph
            clean_text = line["text"].replace("\n", " ").replace("\xa0", " ")
            current_section += " ".join(clean_text.split()) + " "
            # Tag the end of the dialogue
            previous_end = line["start"] + line["duration"]

    # If there's a paragraph left at the end, add it to the list of paragraphs
    if current_section:
        end_time = transcript[-1]["start"] + transcript[-1]["duration"]
        sections.append(
            {
                "text": current_section.strip().replace("\n", " ").replace("\xa0", " "),
                "start_time": start_time,
                "end_time": end_time,
            }
        )
    # Remove empty paragraphs
    sections = [p for p in sections if p["text"]]
    return sections

In [18]:
def junta_transcricao(sections):
    full_transcript=[]
    full_transcript=sections[0]['text'].join(sections[i]['text'] for i in range(0, len(sections)))
    return full_transcript
    

In [19]:
def split_transcricao(long_string, max_length:int=2000):
    parts = []

    # Verifica se a string é maior que o tamanho máximo
    if len(long_string) > max_length:
        start_idx = 0
        while start_idx < len(long_string):
            # Divide a string em partes de tamanho máximo
            part = long_string[start_idx:start_idx + max_length]
            parts.append(part)

            # Atualiza o índice de início para a próxima parte
            start_idx += max_length
    else:
        parts.append(long_string)

    return parts

In [20]:
def imprime_transcricao(parts):
    for i, part in enumerate(parts):
        print(f"Parte {i + 1}:")
        print(part)
        print("-----------------------")

In [21]:
def export_transcricao_to_txt(texto, nome_arquivo):
    try:
        with open(nome_arquivo, 'w', encoding='utf-8') as arquivo:
            arquivo.write(texto)
        print(f"String exportada com sucesso para o arquivo '{nome_arquivo}'.")
    except Exception as e:
        print(f"Erro ao exportar a string para o arquivo: {e}")

In [22]:
def ler_arquivo_csv(caminho_arquivo):
    dicionario_videos = {}
    
    with open(caminho_arquivo, 'r', encoding='utf-8') as arquivo:
        for linha in arquivo:
            id_video, titulo = linha.strip().split(';', maxsplit=1)
            dicionario_videos[id_video] = titulo
            
    return dicionario_videos

Funções objetivo

In [23]:
def exporta_transcricao(video_id:str , path:str, idioma:str='pt', pause_threshold: int=3):
    transcricao=pega_transcricao(video_id, idioma)
    transcricao_tratada=trata_transcricao(transcricao, pause_threshold)
    full_transcricao=junta_transcricao(transcricao_tratada)
    export_transcricao_to_txt(full_transcricao, path)

In [24]:
def imprime_transcricao_partes(video_id:str , idioma:str='pt', pause_threshold: int=3, max_length:int=2000):
    transcricao=pega_transcricao(video_id, idioma)
    transcricao_tratada=trata_transcricao(transcricao, pause_threshold)
    full_transcricao=junta_transcricao(transcricao_tratada)
    imprime_transcricao(split_transcricao(full_transcricao, max_length))
    

In [25]:
def exporta_transcricao_seriada(inputfile: str, outputfolder: str):
    dicionario_videos = ler_arquivo_csv(inputfile)
    log_path = outputfolder + "//log.txt"

    with open(log_path, 'w') as log_file:
        for video_id, arquivo in dicionario_videos.items():
            # Substituir caracteres especiais por underscores na chave do dicionário
            arquivo = arquivo.translate(str.maketrans("/?\|;", '_____'))

            try:
                caminho_arquivo = outputfolder + "//" + arquivo + ".txt"
                exporta_transcricao(video_id, caminho_arquivo)
            except Exception as e:
                mensagem_erro = f"Erro ao exportar a transcrição para o vídeo {video_id}.\n"
                mensagem_erro += f"Erro: {e}\n\n"
                log_file.write(mensagem_erro)

Dados de entrada:
Path do arquivo txt em caso de exportação da transcrição, video ID

In [26]:
#path='YOUR_OUTPUT_PATH'
#video='VIDEO_ID'
#exporta_transcricao(video, path)

In [42]:
import os
caminho = os.path.join(os.getcwd(), 'video_ids', 'playlist_videos.csv')
inputfile=caminho

saida=os.path.join(os.getcwd(), 'transcripts', 'playlist_displinas_dados')
os.makedirs(saida) if not os.path.exists(saida) else None
outputfolder=saida

In [43]:
exporta_transcricao_seriada(inputfile,outputfolder)

String exportada com sucesso para o arquivo 'd:\Romario\Ocupacaoes\ComunidadeDeDados\THIAGO-GPT\transcripts\playlist_displinas_dados//Importância da Estatística descritiva na área de dados #1.txt'.
String exportada com sucesso para o arquivo 'd:\Romario\Ocupacaoes\ComunidadeDeDados\THIAGO-GPT\transcripts\playlist_displinas_dados//Importância dos conceitos de Business intelligence, consultas e SQL dentro da área de dados #2.txt'.
String exportada com sucesso para o arquivo 'd:\Romario\Ocupacaoes\ComunidadeDeDados\THIAGO-GPT\transcripts\playlist_displinas_dados//Importância da visualização de dados na área de dados #3.txt'.
String exportada com sucesso para o arquivo 'd:\Romario\Ocupacaoes\ComunidadeDeDados\THIAGO-GPT\transcripts\playlist_displinas_dados//Importância da Inferência estatística dentro da ciência de dados.txt'.
String exportada com sucesso para o arquivo 'd:\Romario\Ocupacaoes\ComunidadeDeDados\THIAGO-GPT\transcripts\playlist_displinas_dados//O Processo de criação de modelo

In [29]:
#imprime_transcricao_partes(video)